# Network capacity forecasting walkthrough

This notebook uses synthetic link utilization only. The data is generated locally so the workflow stays reproducible without private telemetry.

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import pandas as pd

from scripts.generate_synthetic_data import build_rows
from network_capacity_forecasting.model import train_forecast_model, save_model, load_model
from network_capacity_forecasting.report import forecast_links, build_report, write_report

ROOT = Path.cwd()
DATA_PATH = ROOT / "data" / "synthetic_link_utilization.csv"
MODEL_PATH = ROOT / "artifacts" / "capacity_model.joblib"
REPORT_PATH = ROOT / "reports" / "forecast_summary.md"

plt.style.use("seaborn-v0_8")

## Generate synthetic data

If the sample file is missing, create it from the local generator. The notebook only works with synthetic data.

In [ ]:
if not DATA_PATH.exists():
    frame = build_rows(days=365)
    DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(DATA_PATH, index=False)
else:
    frame = pd.read_csv(DATA_PATH)

frame.head()

## Train the forecast model

Use the synthetic time series to fit a small regression model that predicts next-day utilization.

In [ ]:
result = train_forecast_model(frame)
save_model(MODEL_PATH, result)

print(f"MAE: {result.mae:.2f}")
print(f"RMSE: {result.rmse:.2f}")
print(f"MAPE: {result.mape:.2f}%")
print(f"Model saved to {MODEL_PATH}")

## Evaluate model performance

Check the holdout metrics and the class of forecast errors that matter for capacity planning.

In [ ]:
model_payload = load_model(MODEL_PATH)
forecast = forecast_links(frame, model_payload)

forecast[["link_id", "site", "utilization_pct", "forecast_utilization_pct", "congestion_risk", "upgrade_note"]]

## Feature importance and risk drivers

Use feature importances to see which lag and rolling signals drive the forecast most strongly.

In [ ]:
feature_importance = pd.Series(result.model.feature_importances_, index=result.model.feature_names_in_).sort_values(ascending=False)
feature_importance.head(7)

## Generate a forecast report

Create a short planning note for the operations and business teams.

## Visualize actual versus forecast

A simple line chart is enough for the notebook review copy.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
for link_id, subset in frame.groupby("link_id"):
    subset = subset.sort_values("date")
    ax.plot(pd.to_datetime(subset["date"]), subset["utilization_pct"], label=f"actual {link_id}")

for link_id, subset in forecast.groupby("link_id"):
    ax.scatter(subset["date"], subset["forecast_utilization_pct"], label=f"forecast {link_id}")

ax.set_title("Synthetic utilization and next-day forecast")
ax.set_ylabel("Utilization (%)")
ax.legend(ncol=2, fontsize=8)
fig.tight_layout()

In [ ]:
report_text = build_report(forecast, {"mae": result.mae, "rmse": result.rmse, "mape": result.mape})
write_report(REPORT_PATH, report_text)

artifacts_dir = ROOT / "reports"
artifacts_dir.mkdir(parents=True, exist_ok=True)
forecast.to_csv(artifacts_dir / "forecast_predictions.csv", index=False)
(artifacts_dir / "forecast_metrics.json").write_text(
    json.dumps({"mae": result.mae, "rmse": result.rmse, "mape": result.mape}, indent=2),
    encoding="utf-8",
)
print(f"Report saved to {REPORT_PATH}")

## Validate the notebook workflow

Confirm the full path runs without external dependencies.

In [ ]:
assert DATA_PATH.exists()
assert MODEL_PATH.exists()
assert REPORT_PATH.exists()
assert not forecast.empty
print("Notebook workflow completed successfully")